# Bài tập Buổi 7 - Logistic Regression & KNN

## Phần 1: Logistic Regression với bộ dữ liệu Titanic

**Sinh viên thực hiện:** Nguyễn Bá Quốc Long

---


## 0. Chuẩn bị môi trường & Import Thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


## 1. Tải dữ liệu (đã cho)


In [167]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. EDA

In [168]:
leaky = ['alive', 'who', 'adult_male', 'class', 'embark_town', 'alone', 'deck']   # điền danh sách cột cần bỏ (chỉ những cột có trong df)
df = df.drop(columns = leaky)
display(f'Shape: {df.shape}')
display(df.info())
display(df.isnull().sum())
display(df.describe())
display(df.describe(include=["object"]))


'Shape: (891, 8)'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   sex       891 non-null    object 
 3   age       714 non-null    float64
 4   sibsp     891 non-null    int64  
 5   parch     891 non-null    int64  
 6   fare      891 non-null    float64
 7   embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


None

survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


,sex,embarked
count,891,889
unique,2,3
top,male,S
freq,577,644


## 3. Chia tập

In [169]:
X = df.drop(columns = 'survived').copy()
y = df['survived'].copy()
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, random_state=42, test_size = 0.15, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, random_state=42, test_size = 0.15/0.85, stratify=y_tmp)

In [ ]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("onehot", OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_test_t = preprocess.transform(X_test)

## 4. Linear Regression

In [171]:
linear_fit = LinearRegression().fit(X_train_t, y_train)
y_pred_linear = (linear_fit.predict(X_test_t)>= 0.5).astype(int)

In [172]:
accuracy_linear = accuracy_score(y_test, y_pred_linear)
precision_linear = precision_score(y_test, y_pred_linear)
recall_linear = recall_score(y_test, y_pred_linear)
f1_linear = f1_score(y_test, y_pred_linear)
confusion_linear = confusion_matrix(y_test, y_pred_linear)
print(f'Accuracy: {accuracy_linear:.4f} | Precision: {precision_linear:.4f} | Recall: {recall_linear:.4f} | F1: {f1_linear:.4f}')
print(confusion_linear)

Accuracy: 0.7612 | Precision: 0.7021 | Recall: 0.6471 | F1: 0.6735
[[69 14]
 [18 33]]


## 5. Logistic Regression

In [173]:
logis_fit = LogisticRegression().fit(X_train_t, y_train)
y_pred_logis = logis_fit.predict(X_test_t)

In [174]:
accuracy_logis = accuracy_score(y_test, y_pred_logis)
precision_logis = precision_score(y_test, y_pred_logis)
recall_logis = recall_score(y_test, y_pred_logis)
f1_logis = f1_score(y_test, y_pred_logis)
confusion_logis = confusion_matrix(y_test, y_pred_logis)
print(f'Accuracy: {accuracy_logis:.4f} | Precision: {precision_logis:.4f} | Recall: {recall_logis:.4f} | F1: {f1_logis:.4f}')
print(confusion_logis)

Accuracy: 0.7761 | Precision: 0.7442 | Recall: 0.6275 | F1: 0.6809
[[72 11]
 [19 32]]


## 6. So sánh Linear và Logistic Regression

Nhìn chung, các chỉ số đánh giá trừ Recall của Logistic Regression đều lớn hơn Linear Regression. Như vậy, Logistic Regression phù hợp hơn đối với bài toán phân loại hành khách sống sót.

---